In [1]:
# ─────────────────────────────────────────────
# [C1] ⚙️ 데이터 준비
# 최초 1회 다운로드 → data/ 폴더에 저장 (이후 오프라인)
# ─────────────────────────────────────────────
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def fetch_uci(url, zip_name, member):
    """UCI 정적 저장소의 zip을 내려받아 data/에 풀고, CSV 경로를 돌려줍니다."""
    csv_path = DATA_DIR / member
    if not csv_path.exists():
        zip_path = DATA_DIR / zip_name
        if not zip_path.exists():
            print(f"내려받는 중… {zip_name}")
            urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extract(member, DATA_DIR)
    return csv_path

shoppers = pd.read_csv(fetch_uci(
    "https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip",
    "online_shoppers.zip", "online_shoppers_intention.csv"))

NUM_COLS = ["Administrative", "Administrative_Duration", "Informational",
            "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
            "BounceRates", "ExitRates", "PageValues", "SpecialDay"]
CAT_COLS = ["Month", "OperatingSystems", "Browser", "Region",
            "TrafficType", "VisitorType", "Weekend"]

y = shoppers["Revenue"].astype(int)

print(f"수치형 {len(NUM_COLS)}개 · 범주형 {len(CAT_COLS)}개 · 타깃 1개")
print("범주별 값 개수:", {c: shoppers[c].nunique() for c in CAT_COLS})
print("\n→ 준비 완료. 이제 여러분 차례입니다.")

내려받는 중… online_shoppers.zip
수치형 10개 · 범주형 7개 · 타깃 1개
범주별 값 개수: {'Month': 10, 'OperatingSystems': 8, 'Browser': 13, 'Region': 9, 'TrafficType': 20, 'VisitorType': 3, 'Weekend': 2}

→ 준비 완료. 이제 여러분 차례입니다.


[문제 1]
1) ColumnTransformer를 만듭니다.
   - 수치형: "passthrough"
   - 범주형: OneHotEncoder(handle_unknown="ignore", min_frequency=20)
2) 그것과 지난 순서의 모델을 Pipeline으로 묶습니다.
   RandomForestClassifier(n_estimators=300, min_samples_leaf=20)
3) 5겹 CV로 F1과 AP(`average_precision`)를 측정하고, 수치형 10개만 썼을 때와 비교합니다.
4) 변환 후 피처가 몇 개가 됐는지 출력합니다.

In [ ]:
# [C2] 문제 1. 범주형 7개 열을 `Pipeline`으로 푼다
# ⌨️ 문제 1 — ColumnTransformer + Pipeline으로 범주형 개방
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# 여기에 코드를 작성하세요 (ColumnTransformer → Pipeline → CV 비교 → 피처 수)

# ColumnTransformer
pre = ColumnTransformer([("num", "passthrough", NUM_COLS), #수치형
                   ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT_COLS)]) #범주형

# Pipeline 
pipe = Pipeline([("pre", pre),
                     ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=20))]) 

# CV 비교
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
s = cross_validate(pipe, df[base_num + ["embarked"]], y, cv=cv, scoring="accuracy")


